# 10 - Capstone: End-to-End IQ Exploration

This notebook integrates all prior skills into one cohesive workflow.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Step 1: Load / Generate IQ Data

In a real scenario, you'd load from a binary file with `np.fromfile()`. Here we generate synthetic IQ data.

In [ ]:
np.random.seed(42)
n_samples = 8192
sample_rate = 2.4e6  # 2.4 MHz (e.g., Bluetooth/WiFi band)
center_freq = 2.4e9  # 2.4 GHz

t = np.arange(n_samples) / sample_rate

# Synthetic QPSK-like signal with frequency offset and noise
symbol_rate = 1e6
symbols = np.random.choice([-1, 1], size=n_samples)
iq = symbols * np.exp(2j * np.pi * 50e3 * t) + 0.15 * (np.random.randn(n_samples) + 1j * np.random.randn(n_samples))

print(f"Data loaded/generated: {n_samples} samples")

## Step 2: Document Shape, Dtype, and Metadata

In [ ]:
# DOCUMENTATION BLOCK
# ==================
# Variable: iq
# Shape: (8192,) — 1D complex array
# Dtype: complex128
# Dimensions meaning: [time samples]
# Sample rate: 2.4 MHz (2.4e6 samples/sec)
# Center frequency: 2.4 GHz
# Units: Volts (assumed)
# Notes: Synthetic QPSK-like signal with 50 kHz offset + AWGN
# ==================

print(f"Shape: {iq.shape}")
print(f"Dtype: {iq.dtype}")
print(f"Sample rate: {sample_rate/1e6:.1f} MHz")
print(f"Center frequency: {center_freq/1e9:.1f} GHz")
print(f"Duration: {n_samples/sample_rate*1e6:.1f} us")

## Step 3: Time-Domain Visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(t * 1e6, iq.real, 'b-', linewidth=0.5)
axes[0].set_ylabel('I (Real)')
axes[0].set_title('I Component')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t * 1e6, iq.imag, 'r-', linewidth=0.5)
axes[1].set_ylabel('Q (Imaginary)')
axes[1].set_title('Q Component')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t * 1e6, np.abs(iq), 'g-', linewidth=0.5)
axes[2].set_ylabel('|IQ|')
axes[2].set_xlabel('Time (\u03bcs)')
axes[2].set_title('Magnitude')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 4: Constellation Diagram

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(iq[::5].real, iq[::5].imag, s=3, alpha=0.3)
plt.title('Constellation Diagram')
plt.xlabel('I (Real)')
plt.ylabel('Q (Imaginary)')
plt.gca().set_aspect('equal')
plt.grid(True, alpha=0.3)
plt.xlim(-3, 3)
plt.ylim(-3, 3)
plt.tight_layout()
plt.show()

## Step 5: Compute Signal Power (Two Methods)

In [ ]:
# Method 1: Direct complex magnitude
power_m1 = np.mean(np.abs(iq) ** 2)

# Method 2: I/Q decomposition
power_m2 = np.mean(iq.real ** 2 + iq.imag ** 2)

print(f"Method 1 (|IQ|^2):  {power_m1:.6f}")
print(f"Method 2 (I^2+Q^2): {power_m2:.6f}")
print(f"Difference:         {abs(power_m1 - power_m2):.2e}")
print(f"Match: {np.isclose(power_m1, power_m2)}")

## Step 6: Find and Fix the Axis Bug

The following code has a subtle axis-related bug. Find it and fix it.

In [ ]:
# BUGGY CODE
def compute_channel_powers(data):
    """Compute power for each channel."""
    # Reshape into I/Q columns
    iq_matrix = data.reshape(2, -1)  # <-- BUG HERE
    
    power_i = np.mean(iq_matrix[0] ** 2)
    power_q = np.mean(iq_matrix[1] ** 2)
    
    return power_i, power_q

p_i_buggy, p_q_buggy = compute_channel_powers(iq)
print(f"Buggy I power: {p_i_buggy:.6f}")
print(f"Buggy Q power: {p_q_buggy:.6f}")
print(f"\nDirect I power: {np.mean(iq.real**2):.6f}")
print(f"Direct Q power: {np.mean(iq.imag**2):.6f}")

### Your Fix

Write the corrected function below:

In [ ]:
# YOUR FIX HERE
pass

<details>
<summary>Solution</summary>

The bug is `data.reshape(2, -1)` which flattens the complex array into 2 parts incorrectly. Instead, use:

```python
def compute_channel_powers(data):
    power_i = np.mean(data.real ** 2)
    power_q = np.mean(data.imag ** 2)
    return power_i, power_q
```
</details>

## Step 7: Pass-Criteria Check

Run the verification below. All checks should pass.

In [ ]:
# Pass criteria
checks = []

# Check 1: Correct shape documented
checks.append(("Shape is (8192,)", iq.shape == (8192,)))

# Check 2: Correct dtype
checks.append(("Dtype is complex", np.issubdtype(iq.dtype, np.complexfloating)))

# Check 3: Power methods match
checks.append(("Power methods match", np.isclose(power_m1, power_m2)))

# Check 4: Power values are reasonable (> 0)
checks.append(("Power is positive", power_m1 > 0))

# Check 5: I and Q have similar power (for this signal)
checks.append(("I and Q power similar", np.isclose(np.mean(iq.real**2), np.mean(iq.imag**2), rtol=0.5)))

print("Pass Criteria:")
for desc, passed in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {desc}")

all_pass = all(p for _, p in checks)
print(f"\nOverall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

## Capstone Reflection

Write a one-paragraph capstone description addressing:

1. What does the signal represent?
2. How did you verify your processing was correct?
3. What did you learn about array axes and dimensions?

**Your capstone description:**






## Summary

In this capstone notebook, we integrated all prior skills:

1. **Loading data** — generated synthetic IQ samples with known properties
2. **Documentation** — recorded shape, dtype, sample rate, and metadata
3. **Time-domain visualization** — plotted I, Q, and magnitude vs. time
4. **Constellation diagram** — scatter-plotted I vs. Q to visualize modulation
5. **Signal power** — computed using two independent methods and verified consistency
6. **Debugging** — identified and fixed a subtle axis-related bug

These skills form the foundation for working with real SDR/IQ data in signal processing pipelines.